In [12]:
import os

In [13]:
%pwd

'd:\\Text-Summerizer-project'

In [14]:
os.chdir("../")

In [15]:
%pwd

'd:\\'

In [26]:
import os

os.chdir(r"D:\Text-Summerizer-project")

print(os.getcwd())

D:\Text-Summerizer-project


In [28]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: Path

In [29]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [30]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        
        create_directories([self.config.artifacts_root])
        
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        
        create_directories([config.root_dir])
        
        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            tokenizer_name= config.tokenizer_name
        )
        
        return data_transformation_config

In [32]:
import os
from textSummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

In [33]:
from transformers import AutoTokenizer
import os
from datasets import load_from_disk

class DataTransformation:
    def __init__(self, config):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)

    def convert_examples_to_features(self, example_batch):

        # Input (dialogue)
        input_encodings = self.tokenizer(
            example_batch['dialogue'],
            max_length=512,
            truncation=True,
            padding="max_length"
        )

        # Target (summary) - modern way
        target_encodings = self.tokenizer(
            text_target=example_batch['summary'],
            max_length=128,
            truncation=True,
            padding="max_length"
        )

        return {
            'input_ids': input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']
        }

    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)

        dataset_samsum_pt = dataset_samsum.map(
            self.convert_examples_to_features,
            batched=True
        )

        dataset_samsum_pt.save_to_disk(
            os.path.join(self.config.root_dir, "samsum_dataset")
        )

In [36]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation_config = DataTransformation(config=data_transformation_config)
    data_transformation_config.convert()
except Exception as e:
    raise e

[2026-06-05 16:50:05,426:INFO:textSummarizerLogger:yaml file: config\config.yaml loaded successfully]
[2026-06-05 16:50:05,429:INFO:textSummarizerLogger:yaml file: params.yaml loaded successfully]
[2026-06-05 16:50:05,437:INFO:textSummarizerLogger:created directory at: artifacts]
[2026-06-05 16:50:05,439:INFO:textSummarizerLogger:created directory at: artifacts/data_transformation]


Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 47477.87 examples/s]
